# 04 - Kiểm thử Module Evaluate
Kiểm tra hàm đo mức nhiễu nền SNR (dB), thuật toán so khớp biên, tính sai số MAE và RMSE (ms).


In [6]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve()
while ROOT_DIR.name != 'main_gk' and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.audio.dataset import load_dataset
from src.segmentation.threshold import calibrate_thresholds
from src.segmentation.segment import segment_signal
from src.evaluate.metrics import (
    estimate_snr_db,
    compute_mae,
    compute_rmse,
    evaluate_signal,
    summarize_metrics,
    print_evaluation_table,
)


### 1. Khảo sát mức nhiễu nền SNR (dB) của các môi trường thu âm


In [7]:
train_signals = load_dataset(ROOT_DIR / 'TinHieuHuanLuyen')
test_signals = load_dataset(ROOT_DIR / 'TinHieuKiemThu')

print('--- TẬP HUẤN LUYỆN ---')
for s in train_signals:
    print(f'{s.name:<12} | Phone={s.is_phone!s:<5} | SNR = {estimate_snr_db(s):5.1f} dB')

print('--- TẬP KIỂM THỬ ---')
for s in test_signals:
    print(f'{s.name:<12} | Phone={s.is_phone!s:<5} | SNR = {estimate_snr_db(s):5.1f} dB')


--- TẬP HUẤN LUYỆN ---
phone_F1     | Phone=True  | SNR =  23.8 dB
phone_M1     | Phone=True  | SNR =  23.3 dB
studio_F1    | Phone=False | SNR =  38.8 dB
studio_M1    | Phone=False | SNR =  37.8 dB
--- TẬP KIỂM THỬ ---
phone_F2     | Phone=True  | SNR =  24.8 dB
phone_M2     | Phone=True  | SNR =  27.0 dB
studio_F2    | Phone=False | SNR =  49.3 dB
studio_M2    | Phone=False | SNR =  37.7 dB


### 2. Đánh giá chi tiết sai số biên MAE và RMSE cho tập kiểm thử


In [8]:
thresholds = calibrate_thresholds(train_signals)
metrics_list = []

for sig in test_signals:
    th = thresholds.get_threshold('binary', sig.is_phone)
    res = segment_signal(sig, th, algorithm='binary', min_silence_ms=200.0)
    met = evaluate_signal(sig, res.predicted_boundaries, algorithm='binary', threshold=th)
    metrics_list.append(met)

print_evaluation_table(metrics_list)


Tên File       | Môi trường | SNR (dB) | Thuật toán | Ngưỡng   | MAE (ms)   | RMSE (ms)  | Số biên (Pred/GT)
---------------------------------------------------------------------------------------------------
phone_F2       | Phone      | 24.8     | binary     | 0.0008   | 55.00      | 77.78      | 5/2
phone_M2       | Phone      | 27.0     | binary     | 0.0008   | 5.00       | 7.07       | 4/2
studio_F2      | Studio     | 49.3     | binary     | 0.0006   | 0.00       | 0.00       | 2/2
studio_M2      | Studio     | 37.7     | binary     | 0.0006   | 10.00      | 10.00      | 2/2

=== TỔNG HỢP SAI SỐ TRUNG BÌNH ===
• Phone  : MAE TB =  30.00 ms | RMSE TB =  42.43 ms (Số file: 2)
• Studio : MAE TB =   5.00 ms | RMSE TB =   5.00 ms (Số file: 2)
• Overall: MAE TB =  17.50 ms | RMSE TB =  23.71 ms (Số file: 4)
